# Stage 2c — Deep Multi-Year Reference (fix self-subtraction)
Build a proper deep reference for field 468/c03/q2 from epochs spanning **years**,
excluding the science epoch's season, so the diff no longer cancels its own sources.
Reuses ztf_image_grabbing (download) + stage1_alignment (align) + stage2b (stack) + ois (diff).



## Rung 0 — Setup

In [ ]:
import sys, os
from pathlib import Path

# config.py lives at the project root, not in notebook/ — put the root on sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # notebooks/ -> repo root
sys.path.insert(0, PROJECT_ROOT)

import config                       # FIRST — loads .env → $ZTFDATA + the np.in1d shim
from ztfquery import query          # now ztfquery sees the right download folder
from ztfquery.io import LOCALSOURCE # = your ztfdata/ path

import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from astropy.time import Time       # for obsjd ↔ calendar-year conversion in Rung 1-2

print("ZTFDATA:", LOCALSOURCE)      # should print your ztfdata/ path


## Rung 1 - Multi-year metadata query (grid-pinned)
Same `load-metadata` as ztf_image_grabbing, but pin the grid in the SQL (field=468, ccdid=3, qid=2) instead of discovering it. No date bound  - the archive already spans yeras; the 1-month window came only from head(5).

In [ ]:
TARGET_RA, TARGET_DEC, BOX = 150.0, 2.0, 0.01

zquery = query.ZTFQuery()
zquery.load_metadata(
    radec=[TARGET_RA, TARGET_DEC],
    size=BOX,
    sql_query="fid=2 AND field=468 AND ccdid=3 AND qid=2", # pin the grid
)
meta = zquery.metatable
clean = meta[meta["infobits"] == 0].sort_values("obsjd").reset_index(drop=True)
print(f"Clean epochs on field 468/c03/q2: {len(clean)} of {len(meta)}")

# prove it spans years: convert obsjd -> calendar year, count per year
years = Time(clean["obsjd"].values, format="jd").to_value("decimalyear").astype(int)
import collections
print("clean epochs per year:", dict(sorted(collections.Counter(years).items())))


## Rung 2 — Choose science + reference epochs
Science = our existing 2018-03-22 epoch (we have its Stage-3 catalog + answer key).
Reference = ~40 sharpest clean epochs from 2020-2024 (years apart from 2018 →
no self-subtraction). Assert none fall within ±60 days of the science epoch.


In [ ]:
SCIENCE_OBSJD = 2458199.7732755
N_REF = 40

# reference pool: exclude the science epoch's YEAR (2018) entirely - the self-sub gard
ref_year = Time(clean["obsjd"].values, format="jd").to_value("decimalyear").astype(int)
pool = clean[(ref_year >= 2020) & (ref_year <= 2024)].copy()

# prefer the sharpest frames (lowest seeing) -> crisp template
pool = pool.sort_values("seeing").reset_index(drop=True)
ref_set = pool.head(N_REF).reset_index(drop=True)

#self-subtraction guard: nothing within +- 60 days of the science epoch
gap = np.abs(ref_set["obsjd"].values - SCIENCE_OBSJD)
assert gap.min() > 60, f"a reference epoch is only {gap.min():.0f} days from science!"

print(f"reference set: {len(ref_set)} epochs")
print(f"  seeing range: {ref_set['seeing'].min():.2f}–{ref_set['seeing'].max():.2f} arcsec")
print(f"  year span:    {Time(ref_set['obsjd'].min(), format='jd').iso[:10]} "
      f"→ {Time(ref_set['obsjd'].max(), format='jd').iso[:10]}")
print(f"  closest to science epoch: {gap.min():.0f} days (guard: >60) ✓")


## Rung 3 — Download the reference epochs
Same download_data call as ztf_image_grabbing Cell 13, just with the 40 reference
row indexes. ~1.6 GB / several minutes. First call may prompt IRSA login (cached).

In [ ]:
# map the chosen reference obsjd values back to metatable row indexes
chosen = meta.index[meta["obsjd"].isin(ref_set["obsjd"])].tolist()
print(f"downloading {len(chosen)} reference sciimg.fits ...")

zquery.download_data(
    "sciimg.fits",
    indexes=chosen,
    show_progress=True,
    nprocess=1,
)
print("download finished.")


## Rung 4 — Align the 40 reference frames onto one grid
Reuse stage1_alignment: reproject every reference frame onto the SHARPEST frame's
WCS. Critical: build the file list from ref_set (the 40 reference obsjds), NOT a
blind glob — a blind glob would pull the 2018 science frame into its own template
and re-create self-subtraction. Write to a dedicated aligned_deepref/ dir.

In [ ]:
from reproject import reproject_interp
import glob

# ref_set rows carry filefracday, which IS the filename stamp (ztf_<filefracday>_...).
# Match on that directly — no obsjd arithmetic.
ref_stamps = {str(int(x)) for x in ref_set["filefracday"]}
print(f"ref stamps: {len(ref_stamps)} (e.g. {sorted(ref_stamps)[:2]})")

all_sci = glob.glob(str(Path(LOCALSOURCE) / "sci" / "**" / "*sciimg.fits"), recursive=True)
ref_files = []
for f in all_sci:
    stamp = Path(f).name.split("_")[1]          # ztf_<filefracday>_000468_...
    if stamp in ref_stamps and not stamp.startswith("2018"):
        ref_files.append(f)

print(f"reference files selected: {len(ref_files)} (expect {len(ref_set)})")
assert len(ref_files) > 0, "still zero — check filefracday matching"
assert all(not Path(f).name.split('_')[1].startswith("2018") for f in ref_files), "2018 leaked in!"


In [ ]:
# Load each reference frame + its WCS, pick the SHARPEST as the target grid,
# reproject all 40 onto it. Same as stage1_alignment Cells 5/7/9.
ref_images = []
for f in sorted(ref_files):
    with fits.open(f) as hdul:
        data = hdul[0].data.astype(float)
        wcs = WCS(hdul[0].header)
        seeing = hdul[0].header.get("SEEING")
    ref_images.append((data, wcs, Path(f).name, seeing))

seeings = [s for *_, s in ref_images]
ref_idx = int(np.argmin(seeings))
ref_data, ref_wcs, ref_name, _ = ref_images[ref_idx]
ref_shape = ref_data.shape
print(f"grid reference: {ref_name}  seeing={seeings[ref_idx]:.2f}  shape={ref_shape}")

aligned = []
for data, wcs, name, _ in ref_images:
    array, _ = reproject_interp((data, wcs), ref_wcs, shape_out=ref_shape)
    aligned.append((array, name))
print(f"aligned {len(aligned)} frames onto the common grid.")


## Rung 5 - Stack the deep template (nanmedian over 40 frames)
Reuse stage2b Cell 3: np.nanmedian across the 40-frame stack. Median votes out single-epoch movers and fills reproject-edge NaNs from other frames. Measure the blank-patch noise vs the old 4-frame template - deeper stack should be ~sqrt(40/4)=3x quieter. save to aligned_deepref/deep_template_multiyear.fits

In [ ]:
# stack the 40 aligned reference frames -> deep template
stack = np.stack([a for a, _ in aligned])
deep_template = np.nanmedian(stack, axis=0)
print(f"deep template: {deep_template.shape}, NaNs: {np.isnan(deep_template).sum()}")

#measure noise on a blank sky patch (away from bright sources) - the depth proof.pool
#compare to the OLD 4-frame template if it's on disk. 
patch = deep_template[1500:1700, 1500:1700]
new_std = np.nanstd(patch)
print(f"deep-template blank-patch std (40 frames): {new_std:.2f}")

old_path = Path(LOCALSOURCE) / "aligned" / "deep_template.fits" 
if old_path.exists():
    old = fits.getdata(str(old_path))
    old_std = np.nanstd(old[1500:1700, 1500:1700])
    print(f"old 4-frame template blank-patch std:      {old_std:.2f}")
    print(f"→ noise reduction: {old_std/new_std:.2f}x  (expect ~3x from √(40/4))")

# save the deep template carrying the reference WCS
out_dir = Path(LOCALSOURCE) / "aligned_deepref"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "deep_template_multiyear.fits"
fits.PrimaryHDU(data=deep_template.astype(np.float32),
                header=ref_wcs.to_header()).writeto(out_path, overwrite=True)
print(f"saved → {out_path}")


In [ ]:
from astropy.stats import sigma_clipped_stats

# proper noise estimate: sigma-clip removes stars, leaving the true sky noise floor
for name, img in [("deep (40-frame)", deep_template),
                  ("old (4-frame)", fits.getdata(str(Path(LOCALSOURCE) /
                   "ztf_own_pipeline_data_testing_small_scale" / "deep_template.fits")))]:
    mean, med, std = sigma_clipped_stats(img[500:2500, 500:2500], sigma=3.0)
    print(f"{name:16s}  sky median={med:8.2f}  sigma-clipped noise={std:.2f}")


## Rung 6 — Difference the science epoch against the deep template (ois)
Load the 2018 science frame, align it to the deep-template grid, then
ois.optimal_system solves for the kernel + photometric scale that matches the
template to the science frame and subtracts. THE test: at a known source centroid,
does the diff show a clean point source (survived) instead of ~0 (self-subtracted)?


In [ ]:
import ois

# science frame = our 2018-03-22 epoch. Align it onto the deep-template grid.
sci_path = glob.glob(str(Path(LOCALSOURCE) / "sci" / "2018" / "0322" / "**" /
                          "*sciimg.fits"), recursive=True)[0]
with fits.open(sci_path) as h:
    sci_raw, sci_wcs = h[0].data.astype(float), WCS(h[0].header)
sci_aligned, _ = reproject_interp((sci_raw, sci_wcs), ref_wcs, shape_out=ref_shape)
print(f"science aligned onto deep-template grid: {sci_aligned.shape}")

# work on the central 1000x1000 (NaN-free, matches Stage-2/3 region)
from astropy.nddata import Cutout2D
cen = (sci_aligned.shape[1] // 2, sci_aligned.shape[0] // 2)
sci_c  = np.nan_to_num(Cutout2D(sci_aligned,   cen, 1000).data)
tmpl_c = np.nan_to_num(Cutout2D(deep_template, cen, 1000).data)

# ois: matches template→science (kernel + flux scale), then subtracts. diff = [0].
diff_deep = ois.optimal_system(image=sci_c, refimage=tmpl_c,
                               kernelshape=(11, 11), method="Bramich")[0]

mean, med, std = sigma_clipped_stats(diff_deep, sigma=3.0)
print(f"deep diff residual noise (sigma-clipped std): {std:.2f}  (ZTF official ≈ 10.9)")



## Rung 6b — THE test: does a real source survive the diff?
Self-subtraction showed up as ~0 at a source centroid. Over-subtraction would
too. A HEALTHY diff shows a clear point source (strong +/- deviation). Check the
brightest Stage-3 detections' centroids in the new deep diff vs the sky noise.


In [ ]:
from astropy.table import Table

# load our Stage-3 catalog → the centroids of real detections (same 1000x1000 region)
cat = Table.read(str(Path(LOCALSOURCE) /
    "ztf_own_pipeline_data_testing_small_scale" / "catalog" /
    "ztf_20180322273264_000468_zr_c03_o_q2_stage3_catalog.ecsv"))

# the catalog x/y are in the central-1000 frame already (Stage-3 worked on the cutout)
top = cat[np.argsort(-cat["snr"])][:8]   # 8 highest-S/N detections
print(f"sky noise floor in deep diff: {std:.2f}\n")
print(f"{'row':<12}{'snr':>7}{'diff@center':>13}{'|dev|/noise':>13}")
for r in top:
    x, y = int(round(r["x_centroid"])), int(round(r["y_centroid"]))
    if 0 <= y < diff_deep.shape[0] and 0 <= x < diff_deep.shape[1]:
        val = diff_deep[y, x]
        print(f"{str(r['label']):<12}{r['snr']:>7.1f}{val:>13.1f}{abs(val)/std:>13.1f}")


## Rung 7 — Re-cut triplets from the deep diff + re-score braai (the payoff)
Cut (3,63,63) [sci, ref, diff] stamps at each catalog centroid, using the NEW
deep-grid channels: sci=sci_c, ref=deep template, diff=diff_deep — all the same
1000x1000 on the same grid. Score braai on each, grade recall vs the answer key's
63 ztf_rb reals. Prediction: recall climbs from ~0.05 toward the native range.


In [ ]:
# all three channels on the SAME deep-template grid, same central 1000x1000
sci_ch  = sci_c                                   # science, aligned to deep grid (Rung 6)
ref_ch  = tmpl_c                                  # the deep template (Rung 6 cutout)
diff_ch = diff_deep                               # the clean ois diff (Rung 6)

STAMP = 63
new_dir = Path(LOCALSOURCE) / "ztf_own_pipeline_data_testing_small_scale" / "cutouts_deepref"
new_dir.mkdir(parents=True, exist_ok=True)

def cut3(x, y):
    kw = dict(size=STAMP, mode="partial", fill_value=0.0)
    s = Cutout2D(sci_ch,  (x, y), **kw).data
    r = Cutout2D(ref_ch,  (x, y), **kw).data
    d = Cutout2D(diff_ch, (x, y), **kw).data
    return np.stack([s, r, d], axis=0)

for row in cat:
    trip = cut3(row["x_centroid"], row["y_centroid"])
    fname = f"src_{row['sign']:+d}_{int(row['label']):03d}.npy".replace("+","p").replace("-","m")
    np.save(new_dir / fname, trip)
print(f"saved {len(cat)} deep-ref triplets → {new_dir}")


In [ ]:
from reproject import reproject_interp
from astropy.nddata import Cutout2D

# --- deep template → official-diff grid (so old centroids are valid) ---
tmpl_path = str(Path(LOCALSOURCE) / "aligned_deepref" / "deep_template_multiyear.fits")
tmpl      = fits.getdata(tmpl_path).astype(float)
tmpl_wcs  = WCS(fits.getheader(tmpl_path))

OFF       = str(Path(LOCALSOURCE) / "difference" /
    "ztf_20180322273264_000468_zr_c03_o_q2_official_diff.fits")
diff_wcs  = WCS(fits.getheader(OFF))
off_shape = fits.getdata(OFF).shape

tmpl_off, _ = reproject_interp((tmpl, tmpl_wcs), diff_wcs, shape_out=off_shape)
cen = (off_shape[1] // 2, off_shape[0] // 2)
tmpl_off_c = np.nan_to_num(Cutout2D(tmpl_off, cen, 1000).data)
print(f"deep template on official grid: {tmpl_off_c.shape}, NaN={np.isnan(tmpl_off_c).sum()}")

# --- re-cut triplets: OLD sci + NEW deep-template ref + OLD (official) diff ---
old_dir = Path(LOCALSOURCE) / "ztf_own_pipeline_data_testing_small_scale" / "cutouts"
new_dir = Path(LOCALSOURCE) / "ztf_own_pipeline_data_testing_small_scale" / "cutouts_deepref_v2"
new_dir.mkdir(parents=True, exist_ok=True)

for row in cat:
    row_id = os.path.splitext(os.path.basename(str(row["stamp_path"])))[0]
    old = np.load(old_dir / f"{row_id}.npy")                 # [0]=sci, [1]=ref, [2]=diff
    newref = Cutout2D(tmpl_off_c, (row["x_centroid"], row["y_centroid"]),
                      size=63, mode="partial", fill_value=0.0).data
    np.save(new_dir / f"{row_id}.npy",
            np.stack([old[0], newref, old[2]], axis=0))       # deep ref, keep official diff
print(f"saved {len(cat)} triplets (old sci + deep ref + official diff) → {new_dir.name}")


In [ ]:
from ztf_classification.braai_realbogus import load_braai, make_triplet, braai_score

braai = load_braai(os.path.join(PROJECT_ROOT, "braai", "models"))

key = Table.read(str(Path(LOCALSOURCE) / "labels" /
    "ztf_20180322273264_labeled_set.ecsv"))
reals = key[key["ztf_rb"] > -99]
print(f"grading braai on {len(reals)} real detections\n")

n_pass, scores = 0, []
for row in reals:
    row_id = os.path.splitext(os.path.basename(str(row["stamp_path"])))[0]
    npy = np.load(new_dir / f"{row_id}.npy")          # cutouts_deepref_v2
    trip = make_triplet(npy[0], npy[1], npy[2])
    p = braai_score(braai, trip)
    scores.append(p)
    if p >= 0.5:
        n_pass += 1

scores = np.array(scores)
print(f"=== braai: old sci + DEEP ref + official diff ===")
print(f"  recall (P(real) ≥ 0.5): {n_pass}/{len(reals)} = {n_pass/len(reals):.3f}")
print(f"  median P(real): {np.median(scores):.3f}   max: {scores.max():.3f}")
print(f"\n  old cutouts (all-old channels):   recall 0.048")
print(f"  native ZTF stamps (ceiling):      recall 0.683")


## Rung A — 2020 science pivot: download science + 2021-24 reference
2018 had only 19 active sources; 2020 has 173. Switch science to a sharp 2020
frame, rebuild the reference from 2021-2024 (exclude 2020 → no self-subtraction).


In [ ]:
from astropy.time import Time

zq2 = query.ZTFQuery()
zq2.load_metadata(radec=[150.0, 2.0], size=0.01,
                  sql_query="fid=2 AND field=468 AND ccdid=3 AND qid=2")
m2 = zq2.metatable
cl = m2[m2["infobits"] == 0].copy()
cl["year"] = Time(cl["obsjd"].values, format="jd").to_value("decimalyear").astype(int)

# reference = sharpest 40 from 2021-2024; science = sharpest 2020 frame ≥120 days from all of them
ref2 = cl[(cl["year"] >= 2021) & (cl["year"] <= 2024)].sort_values("seeing").head(40)
c2020 = cl[cl["year"] == 2020].sort_values("seeing")
SCI2020 = None
for _, cand in c2020.iterrows():
    if np.abs(ref2["obsjd"].values - cand["obsjd"]).min() > 120:
        SCI2020 = cand
        break
assert SCI2020 is not None, "no well-separated 2020 frame found"

print(f"science 2020: filefracday={int(SCI2020['filefracday'])} seeing={SCI2020['seeing']:.2f}")
print(f"reference: {len(ref2)} frames, 2021-2024, seeing {ref2['seeing'].min():.2f}-{ref2['seeing'].max():.2f}")

gap = np.abs(ref2["obsjd"].values - SCI2020["obsjd"])
assert gap.min() > 60, f"ref only {gap.min():.0f} days from science!"
print(f"closest ref to science: {gap.min():.0f} days (>60 ✓)")

want = [int(SCI2020["filefracday"])] + [int(x) for x in ref2["filefracday"]]
idx = m2.index[m2["filefracday"].isin(want)].tolist()
print(f"downloading {len(idx)} frames (1 science + {len(ref2)} reference)...")
zq2.download_data("sciimg.fits", indexes=idx, show_progress=True, nprocess=1)
print("done.")



## Rung B — Align 2021-24 reference + stack deep template (2020 grid)
Reuse Rung 4/5: reproject the 40 reference frames onto the sharpest one, nanmedian
stack. Select files by ref2 filefracday (NOT a blind glob — keeps the 2020 science
frame out of its own template).


In [ ]:
from reproject import reproject_interp

# select the 40 reference files by filefracday (the load-bearing lesson from Rung 4)
ref_stamps2 = {str(int(x)) for x in ref2["filefracday"]}
all_sci = glob.glob(str(Path(LOCALSOURCE) / "sci" / "**" / "*sciimg.fits"), recursive=True)
ref_files2 = [f for f in all_sci
              if Path(f).name.split("_")[1] in ref_stamps2]
sci2020_stamp = str(int(SCI2020["filefracday"]))
assert sci2020_stamp not in {Path(f).name.split("_")[1] for f in ref_files2}, "science leaked!"
print(f"reference files: {len(ref_files2)}/40, science frame excluded ✓")

# load + reproject onto the sharpest reference frame
imgs2 = []
for f in sorted(ref_files2):
    with fits.open(f) as h:
        imgs2.append((h[0].data.astype(float), WCS(h[0].header), h[0].header.get("SEEING")))
ridx = int(np.argmin([s for *_, s in imgs2]))
rdata, rwcs2, _ = imgs2[ridx]
rshape2 = rdata.shape
print(f"grid ref seeing={min(s for *_,s in imgs2):.2f}, shape={rshape2}")

aligned2 = []
for data, wcs, _ in imgs2:
    arr, _ = reproject_interp((data, wcs), rwcs2, shape_out=rshape2)
    aligned2.append(arr)

deep_template_2020ref = np.nanmedian(np.stack(aligned2), axis=0)
from astropy.stats import sigma_clipped_stats
_, _, tstd = sigma_clipped_stats(deep_template_2020ref[500:2500, 500:2500], sigma=3.0)
print(f"deep template (2021-24, 40 frames): noise floor {tstd:.2f}")

# save it
out2 = Path(LOCALSOURCE) / "aligned_deepref" / "deep_template_2020ref.fits"
fits.PrimaryHDU(deep_template_2020ref.astype(np.float32),
                header=rwcs2.to_header()).writeto(str(out2), overwrite=True)
print(f"saved → {out2.name}")


## Rung C — Download ZTF's official 2020 difference image
The professional denoised diff for our 2020 science exposure (against ZTF's own
deep reference). Same scimrefdiffimg product we used for 2018 — the v2 lesson:
use ZTF's diff, not a homemade one.


In [ ]:
# download ZTF's official difference for the 2020 science exposure
idx2020 = m2.index[m2["filefracday"] == int(SCI2020["filefracday"])].tolist()
print(f"downloading official diff for exposure index {idx2020}...")
zq2.download_data("scimrefdiffimg.fits.fz", indexes=idx2020,
                  show_progress=True, nprocess=1)

# locate it on disk (lands next to the sci exposure)
diff2020 = glob.glob(str(Path(LOCALSOURCE) / "sci" / "**" /
                          "*20200518187454*scimrefdiffimg.fits.fz"), recursive=True)
print("official 2020 diff on disk:", diff2020)


## Rung D — Stage 3 on the 2020 diff: detect + measure → new catalog + cutouts
Reuse the Stage-3 rungs (Background2D → detect_sources both signs → deblend →
SourceCatalog) on ZTF's official 2020 diff. Cutout channels: 2020 sci + our
2021-24 DEEP ref + ZTF's official diff (the v2 winning recipe).


In [ ]:
from astropy.stats import SigmaClip
from astropy.nddata import Cutout2D
from photutils.background import Background2D, MedianBackground
from photutils.segmentation import detect_sources, deblend_sources, SourceCatalog

# official 2020 diff: data in HDU 1 (.fz fpack-compressed), carry its WCS
with fits.open(diff2020[0]) as h:
    diff2020_full = h[1].data.astype(float)
    diff2020_wcs  = WCS(h[1].header)

# science (2020) + deep template, reprojected onto the DIFF's grid (so centroids are valid)
sci2020_path = glob.glob(str(Path(LOCALSOURCE) / "sci" / "**" /
    "*20200518187454*sciimg.fits"), recursive=True)[0]
with fits.open(sci2020_path) as h:
    sci2020_full, sci2020_wcs = h[0].data.astype(float), WCS(h[0].header)

sci2020_off, _  = reproject_interp((sci2020_full, sci2020_wcs), diff2020_wcs,
                                   shape_out=diff2020_full.shape)
tmpl2020_off, _ = reproject_interp((deep_template_2020ref, rwcs2), diff2020_wcs,
                                   shape_out=diff2020_full.shape)

# central 1000x1000 (same convention as 2018 Stage 3), carry sliced WCS
cen = (diff2020_full.shape[1] // 2, diff2020_full.shape[0] // 2)
diff_c = Cutout2D(diff2020_full, cen, 1000, wcs=diff2020_wcs)
cut_wcs = diff_c.wcs
diff_img = np.nan_to_num(diff_c.data)
sci_img  = np.nan_to_num(Cutout2D(sci2020_off,  cen, 1000).data)
ref_img  = np.nan_to_num(Cutout2D(tmpl2020_off, cen, 1000).data)
print("cutouts ready:", diff_img.shape, "| diff NaN:", np.isnan(diff_img).sum())

# background + noise map, detect BOTH signs at 3sigma (inclusive, like 2018)
bkg = Background2D(diff_img, box_size=(64,64), filter_size=(3,3),
                   sigma_clip=SigmaClip(3.0), bkg_estimator=MedianBackground())
rms = bkg.background_rms
segm_pos = detect_sources(diff_img,  3*rms, npixels=5)
segm_neg = detect_sources(-diff_img, 3*rms, npixels=5)
np_ = segm_pos.nlabels if segm_pos else 0
nn_ = segm_neg.nlabels if segm_neg else 0
print(f"detected: {np_} positive + {nn_} negative = {np_+nn_}")


In [ ]:
from astropy.table import vstack

def measure(segm, data, sign):
    segm_db = deblend_sources(data, segm, n_pixels=5, n_levels=32, contrast=0.001)
    cols = ["label", "x_centroid", "y_centroid", "sky_centroid",
            "segment_flux", "segment_flux_err", "elongation"]
    cat = SourceCatalog(data, segm_db, wcs=cut_wcs, error=rms).to_table(columns=cols)
    cat["sign"] = sign
    return cat

cat_pos = measure(segm_pos,  diff_img, +1)
cat_neg = measure(segm_neg, -diff_img, -1)
catalog2020 = vstack([cat_pos, cat_neg])
catalog2020["snr"] = catalog2020["segment_flux"] / catalog2020["segment_flux_err"]
print(f"measured {len(catalog2020)} sources | S/N {catalog2020['snr'].min():.1f}–{catalog2020['snr'].max():.1f}")

STAMP = 63
new_dir2020 = Path(LOCALSOURCE) / "ztf_own_pipeline_data_testing_small_scale" / "cutouts_2020"
new_dir2020.mkdir(parents=True, exist_ok=True)

def cut3(img, x, y):
    return Cutout2D(img, (x, y), size=STAMP, mode="partial", fill_value=0.0).data

paths2020, edge2020 = [], []
half = STAMP / 2
H, W = diff_img.shape
for i, row in enumerate(catalog2020):
    x, y = row["x_centroid"], row["y_centroid"]
    trip = np.stack([cut3(sci_img, x, y), cut3(ref_img, x, y), cut3(diff_img, x, y)], axis=0)
    on_edge = (x < half) or (y < half) or (x > W - half) or (y > H - half)
    fname = f"src_{row['sign']:+d}_{i:03d}.npy".replace("+","p").replace("-","m")
    np.save(new_dir2020 / fname, trip)
    paths2020.append(str(new_dir2020 / fname)); edge2020.append(on_edge)

catalog2020["stamp_path"] = paths2020
catalog2020["on_edge"] = edge2020
catalog2020["ra"] = catalog2020["sky_centroid"].ra.deg
catalog2020["dec"] = catalog2020["sky_centroid"].dec.deg

cat_dir = Path(LOCALSOURCE) / "ztf_own_pipeline_data_testing_small_scale" / "catalog"
catalog2020[["label","x_centroid","y_centroid","segment_flux","elongation","snr","sign",
             "ra","dec","stamp_path","on_edge"]].write(
    str(cat_dir / "ztf_20200518187454_stage3_catalog.ecsv"),
    format="ascii.ecsv", overwrite=True)
print(f"saved {len(catalog2020)} triplets + catalog → cutouts_2020/")



## Rung E — Build the answer key (ALeRCE rb cross-match)
Cross-match all 114 detections to ALeRCE for ZTF's own rb (real/bogus) — the label
that grades braai. Reuses crossmatch_labels.label_ztf_alerce. ~114 network calls,
2-4 min. Save so we pay the network cost once.


In [ ]:
from alerce.core import Alerce
from astropy.coordinates import SkyCoord
import astropy.units as u
from ztf_classification.crossmatch_labels import label_ztf_alerce

client = Alerce()

# build SkyCoords from the catalog ra/dec
det_coords = [SkyCoord(row["ra"]*u.deg, row["dec"]*u.deg) for row in catalog2020]
print(f"cross-matching {len(det_coords)} detections to ALeRCE...")

alerce_labels = label_ztf_alerce(det_coords, client, radius_arcsec=5.0)

# attach ztf_rb to the catalog (-99 = no match, like the 2018 answer key convention)
ztf_rb = np.full(len(catalog2020), -99.0)
for i, d in alerce_labels.items():
    if d.get("ztf_rb") is not None:
        ztf_rb[i] = d["ztf_rb"]
catalog2020["ztf_rb"] = ztf_rb

n_matched = int((ztf_rb > -99).sum())
print(f"\nmatched to ALeRCE with an rb: {n_matched}/{len(catalog2020)}")
print(f"  (these are the 'reals' braai will be graded against)")

# save the answer key
key_path = Path(LOCALSOURCE) / "labels" / "ztf_20200518187454_labeled_set.ecsv"
catalog2020.write(str(key_path), format="ascii.ecsv", overwrite=True)
print(f"saved answer key → {key_path.name}")


## Rung F — Score braai on the 2020 cutouts (the payoff)
Score braai on all 82 real detections' triplets (2020 sci + deep ref + official
diff). Grade recall. The busier-year bet: more genuinely-bright-in-2020 sources
→ recall should climb above the 2018 deep-ref 0.143.


In [ ]:
from ztf_classification.braai_realbogus import load_braai, make_triplet, braai_score

braai = load_braai(os.path.join(PROJECT_ROOT, "braai", "models"))

reals2020 = catalog2020[catalog2020["ztf_rb"] > -99]
print(f"grading braai on {len(reals2020)} real detections\n")

n_pass, scores = 0, []
for row in reals2020:
    npy = np.load(row["stamp_path"])                    # (3,63,63): 2020 sci, deep ref, official diff
    p = braai_score(braai, make_triplet(npy[0], npy[1], npy[2]))
    scores.append(p)
    if p >= 0.5:
        n_pass += 1

scores = np.array(scores)
print(f"=== braai on 2020 cutouts (busier year + deep ref + official diff) ===")
print(f"  recall (P(real) ≥ 0.5): {n_pass}/{len(reals2020)} = {n_pass/len(reals2020):.3f}")
print(f"  median P(real): {np.median(scores):.3f}   max: {scores.max():.3f}")
print(f"\n  2018 old cutouts:              recall 0.048")
print(f"  2018 deep-ref v2:              recall 0.143")
print(f"  native ZTF stamps (ceiling):   recall 0.683")


## Rung G — Route our detections through ZTF's NATIVE stamps
The dipole is in OUR sci/ref channels. For any detection with an ALeRCE match,
pull ZTF's own clean (sci,ref,diff) via get_stamps and score braai on THAT. This
is the native-stamp path applied to OUR OWN detections — no homemade cutout.


In [ ]:
from ztf_classification.stamp_classifier import resolve_oid_for_coords

reals2020 = catalog2020[catalog2020["ztf_rb"] >= 0.5]
print(f"routing {len(reals2020)} rb≥0.5 reals through native stamps...\n")

n_native, n_ours, matched = 0, 0, 0
for row in reals2020:
    # our homemade cutout score (the dipole-limited one)
    our_p = braai_score(braai, make_triplet(*np.load(row["stamp_path"])))
    if our_p >= 0.5:
        n_ours += 1

    # ZTF's native stamp for the same object
    oid = resolve_oid_for_coords(client, row["ra"], row["dec"], radius_arcsec=5.0)
    if oid is None:
        continue
    try:
        h = client.get_stamps(oid, format="HDUList")
        nat_p = braai_score(braai, make_triplet(h[0].data, h[1].data, h[2].data))
    except Exception:
        continue
    matched += 1
    if nat_p >= 0.5:
        n_native += 1

print(f"=== our detections, braai recall ===")
print(f"  OUR cutouts (dipole-limited):  {n_ours}/{len(reals2020)} = {n_ours/len(reals2020):.3f}")
print(f"  NATIVE stamps (matched, {matched}): {n_native}/{matched} = {n_native/matched:.3f}")
print(f"\n  → the dipole gap, closed for ALeRCE-matched detections")


## Rung H — Full pipeline on 2020 detections (native-stamp route)
Run the complete Stage-4 pipeline (braai + point-source type + streak) on all 114
real 2020 detections, using ZTF's native stamps where matched (dipole-free). One
verdict per detection at professional quality.


In [ ]:
from ztf_classification.pipeline import (run_braai_gate_native, run_pathB_type,
                                         build_verdict_table, merge_verdict)
from ztf_classification.braai_realbogus import load_braai
from collections import Counter

model = load_braai(os.path.join(PROJECT_ROOT, "braai", "models"))
CUTOUTS_2020 = str(Path(LOCALSOURCE) / "ztf_own_pipeline_data_testing_small_scale" / "cutouts_2020")

# Rung 1: braai gate via NATIVE stamps (falls back to cutout for novel objects)
results = run_braai_gate_native(catalog2020, CUTOUTS_2020, model, client)
src = Counter(r["braai_stamp_source"] for r in results)
n_pass = sum(r["braai_pass"] for r in results)
print(f"braai gate: {n_pass}/{len(results)} passed real  |  stamp source: {dict(src)}")

# Rung 2: point-source TYPE (Path B / ALeRCE) on the passers
results = run_pathB_type(results, client)
typed = [r for r in results if r["braai_pass"] and r.get("pathB_type")]
print(f"typed {len(typed)} passers via ALeRCE")
for r in typed[:10]:
    print(f"  {r['row_id']}: {r['pathB_type']} (P={r.get('pathB_prob', 0):.2f})")


In [ ]:
OUT = str(Path(LOCALSOURCE) / "ztf_own_pipeline_data_testing_small_scale" /
          "stage4_pipeline_verdicts_2020_native.ecsv")

# add the stamp-source audit field into the verdict table build
verdicts = build_verdict_table(results, OUT)

# add the native/cutout column for auditability
import numpy as np
verdicts["braai_stamp_source"] = [r["braai_stamp_source"] for r in results]
verdicts.write(OUT, format="ascii.ecsv", overwrite=True)

print(f"wrote {len(verdicts)} verdicts → {os.path.basename(OUT)}\n")
print("final verdict distribution:")
for v, n in Counter(verdicts["final_verdict"]).most_common():
    print(f"  {v}: {n}")

print("\nbraai-passed detections by stamp source:")
passed = [r for r in results if r["braai_pass"]]
print(f"  native (professional): {sum(r['braai_stamp_source']=='native' for r in passed)}")
print(f"  cutout (dipole-limited): {sum(r['braai_stamp_source']=='cutout' for r in passed)}")


## Rung I — DeepStreaks motion branch on 2020 detections
Cut 144×144 dark-on-gray streak stamps from the 2020 diff cutout at the 60
elongated (elongation>1.5, not on_edge) centroids, score through the DeepStreaks
cascade (.venv_streaks subprocess). Routes: SHORT_NEA / LONG_SATELLITE / COSMIC_RAY
/ BOGUS. Field 468 has no known streakers, so this tests the cascade rejecting noise.


In [ ]:
from ztf_classification.pipeline import build_streak_batch, run_streak_subprocess
from collections import Counter

# streak stamps come from the 2020 DIFF CUTOUT (same 1000x1000 frame the centroids live in)
streak_batch, streak_ids = build_streak_batch(results, catalog2020, diff_img)
print(f"cut {len(streak_ids)} streak stamps (elongated, not on_edge)")

# score through DeepStreaks in the .venv_streaks subprocess (Keras-version wall)
SCRATCH = str(Path(LOCALSOURCE) / "scratch")
streak_results = run_streak_subprocess(streak_batch, streak_ids, SCRATCH)

# merge routes back onto the detection records by row_id
route_by_id = {rid: d["route"] for rid, d in streak_results.items()}
for r in results:
    r["streak_route"] = route_by_id.get(r["row_id"])   # None if not elongated

print("\nstreak routes:", dict(Counter(d["route"] for d in streak_results.values())))


In [ ]:
# rebuild the verdict table now that streak_route is on every record
verdicts = build_verdict_table(results, OUT)
verdicts["braai_stamp_source"] = [r["braai_stamp_source"] for r in results]
verdicts["streak_route"] = [r.get("streak_route") or "" for r in results]
verdicts.write(OUT, format="ascii.ecsv", overwrite=True)

print(f"final pipeline output: {len(verdicts)} verdicts (all 4 branches)\n")
print("verdict distribution:", dict(Counter(verdicts["final_verdict"])))
print(f"elongated → streak-scored: {sum(1 for r in results if r.get('streak_route'))}")
print(f"  (all BOGUS — field 468 has no fast-movers; cascade correctly rejecting noise)")

